# КодФильм: рендер фильма на бесплатном GPU Colab

1. Меню **Среда выполнения → Сменить среду выполнения → T4 GPU → Сохранить**.
2. Нажми **Среда выполнения → Выполнить все** (Ctrl+F9).
3. Ячейка 1 попросит доступ к Google Диску: разреши. Ячейка 2 попросит файл `kodfilm_colab.zip`: выбери его.
4. Держи вкладку открытой, пока идёт рендер (4–6 часов). Кадры сохраняются на Google Диск в папку `kodfilm_out`.
5. Если Colab оборвал сеанс, просто снова нажми **Выполнить все**: рендер продолжится с того места, где остановился.
6. В конце скачается `kodfilm_film.zip`. Пришли его в чат.

In [ ]:
# 1. GPU и Google Диск (на Диске копятся кадры — обрыв сеанса не теряет работу)
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
import os
D = '/content/drive/MyDrive/kodfilm_out'; os.makedirs(D, exist_ok=True)
print('кадры будут здесь:', D)

In [ ]:
# 2. Код студии: kodfilm_colab.zip (берётся с Диска, если уже загружали раньше)
import os, shutil, zipfile
Z = os.path.join(D, 'kodfilm_colab.zip')
if not os.path.exists(Z):
    from google.colab import files
    up = files.upload()
    name = next(iter(up)); shutil.move(name, Z)
shutil.rmtree('/content/kf', ignore_errors=True)
zipfile.ZipFile(Z).extractall('/content/kf')
print(sorted(os.listdir('/content/kf')))

In [ ]:
# 3. Blender 5.2.3 (Python 3.13 через uv) и бесплатные ассеты студии — ~10 минут
!apt-get -qq install -y libxi6 libxxf86vm1 libxfixes3 libxrender1 libsm6 libgl1 libegl1 > /dev/null 2>&1
!pip -q install uv
!uv venv -q -p 3.13 /content/venv
!uv pip install -q -p /content/venv/bin/python bpy==5.2.3 numpy soundfile
!cd /content/kf && bash assets/fetch.sh && /content/venv/bin/python assets/fetch_hq.py 4k
!/content/venv/bin/python -c "import bpy; print('Blender', bpy.app.version_string)"

In [ ]:
# 4. Рендер: Cycles на T4, 1920 px. Уже готовые кадры пропускаются (--resume)
!cd /content/kf && /content/venv/bin/python studio/film.py --script films/hram/script.txt --hero pilgrim --w 1920 --engine cycles --samples 64 --resume --out {D}/frames 2>&1 | grep --line-buffered -E 'resume|^frame|Cycles GPU|Traceback|Error'

In [ ]:
# 5. Упаковка: кадры → frames.mp4 (почти без потерь) + meta.json → kodfilm_film.zip, скачивание
import glob, subprocess
fr = sorted(glob.glob(f'{D}/frames/f_*.png')); print('кадров:', len(fr))
subprocess.run(f'ffmpeg -loglevel error -y -framerate 24 -start_number 1 -i {D}/frames/f_%05d.png -c:v libx264 -preset slow -crf 10 -pix_fmt yuv420p {D}/frames.mp4', shell=True, check=True)
shutil.copy(f'{D}/frames/meta.json', f'{D}/meta.json')
with zipfile.ZipFile(f'{D}/kodfilm_film.zip', 'w') as z:
    z.write(f'{D}/frames.mp4', 'frames.mp4'); z.write(f'{D}/meta.json', 'meta.json')
print('готово:', f'{D}/kodfilm_film.zip', os.path.getsize(f'{D}/kodfilm_film.zip') // 2**20, 'МБ')
from google.colab import files
files.download(f'{D}/kodfilm_film.zip')